# 04 — Hope Architecture

Integrates associative memory (attention), CMS (multi-frequency), and DMGD (meta-learning).

Experiments:
1. TitanBlock standalone — shape & gradient verification
2. Surprise visualization — distribution shift detection
3. HopeBlock CMS test — multi-timescale response
4. Character-level LM on Tiny Shakespeare
5. In-context learning test
6. Self-modification visualization (DMGD meta-params)

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.hope import TitanBlock, HopeBlock, HopeModel, HopeTrainer
from src.data import load_tiny_shakespeare, CharTokenizer, SequenceDataset
from src.utils import set_seed, plot_loss_curves, count_parameters
set_seed(42)

## 1. TitanBlock Standalone — Shape & Gradient Check

In [ ]:
block = TitanBlock(embed_dim=64, n_heads=4)
x = torch.randn(2, 32, 64, requires_grad=True)
out, surprise = block(x)

print(f'Input shape:    {x.shape}')
print(f'Output shape:   {out.shape}')
print(f'Surprise shape: {surprise.shape}')
print(f'Parameters:     {count_parameters(block):,}')

# Gradient flow check
out.sum().backward()
print(f'Input grad norm: {x.grad.norm():.4f}')
assert x.grad is not None and x.grad.abs().sum() > 0
print('Gradient flow: OK')

## 2. Surprise Visualization — Distribution Shift

In [ ]:
# Create a sequence that shifts distribution midway
# First half: token 0 repeated, second half: token 1
torch.manual_seed(42)
model = HopeModel(vocab_size=10, embed_dim=32, n_heads=2, n_layers=2, max_seq_len=100)

# Sequence: [0,0,0,...,0, 5,5,5,...,5] — shift happens at position 50
seq = torch.cat([torch.zeros(50, dtype=torch.long), 5 * torch.ones(50, dtype=torch.long)])
seq = seq.unsqueeze(0)  # (1, 100)

with torch.no_grad():
    logits, surprises = model(seq)

# Average surprise across layers
avg_surprise = torch.stack(surprises).mean(dim=0).squeeze()  # (100,)

plt.figure(figsize=(10, 4))
plt.plot(avg_surprise.numpy())
plt.axvline(x=50, color='r', ls='--', label='Distribution shift')
plt.xlabel('Position')
plt.ylabel('Surprise')
plt.title('Surprise Score — Distribution Shift Detection\n(all 0s then all 5s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('Note: With an untrained model, surprise is high everywhere.')
print('After training, surprise should spike at the distribution shift point.')

## 3. HopeBlock CMS Test — Multi-Timescale Response

In [ ]:
block = HopeBlock(embed_dim=32, n_heads=2, c_base=4)

# Show which levels are active at different steps
for step in [1, 2, 3, 4, 8, 16, 32, 64]:
    active = block.get_active_levels(step)
    print(f'Step {step:3d}: active levels = {active}')

## 4. Character-Level LM on Tiny Shakespeare

In [ ]:
from torch.utils.data import DataLoader
import math

# Load data
try:
    text = load_tiny_shakespeare(max_chars=100_000, data_dir='../data')
except:
    # Fallback: generate synthetic text
    import string
    text = ''.join(np.random.choice(list(string.ascii_lowercase + ' \n'), size=50000))
    print('Using synthetic text (could not download Shakespeare)')

tokenizer = CharTokenizer(text)
tokens = tokenizer.encode(text)
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Total tokens: {len(tokens):,}')

seq_len = 64
dataset = SequenceDataset(tokens, seq_len)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

In [ ]:
# Train Hope model
set_seed(42)
hope_model = HopeModel(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64, n_heads=2, n_layers=2,
    max_seq_len=seq_len, c_base=4,
)
print(f'Hope parameters: {count_parameters(hope_model):,}')
trainer = HopeTrainer(hope_model, lr=3e-4)

hope_losses = []
for epoch in range(3):
    logs = trainer.train_epoch(loader, verbose=True)
    epoch_loss = np.mean([l['loss'] for l in logs])
    hope_losses.extend([l['loss'] for l in logs])
    print(f'Epoch {epoch+1}: avg loss = {epoch_loss:.4f}, PPL = {math.exp(epoch_loss):.1f}')

In [ ]:
# Simple RNN baseline (same param budget)
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        h = self.embed(x)
        h, _ = self.rnn(h)
        return self.head(h)

set_seed(42)
rnn_model = SimpleRNN(tokenizer.vocab_size, 64, 64, 2)
print(f'RNN parameters: {count_parameters(rnn_model):,}')
rnn_opt = torch.optim.Adam(rnn_model.parameters(), lr=3e-4)
rnn_losses = []

for epoch in range(3):
    rnn_model.train()
    epoch_ls = []
    for x, y in loader:
        rnn_opt.zero_grad()
        logits = rnn_model(x)
        loss = nn.CrossEntropyLoss()(logits.view(-1, tokenizer.vocab_size), y.view(-1))
        loss.backward()
        rnn_opt.step()
        rnn_losses.append(loss.item())
        epoch_ls.append(loss.item())
    print(f'RNN Epoch {epoch+1}: avg loss = {np.mean(epoch_ls):.4f}')

# Compare
plot_loss_curves(
    {'Hope': hope_losses, 'GRU': rnn_losses},
    title='Char-Level LM: Hope vs GRU (Tiny Shakespeare)',
)
plt.show()

## 5. In-Context Learning Test

In [ ]:
# Test if the model can pick up simple patterns from context
hope_model.eval()
prompt_text = 'To be, or not to be'
prompt_ids = torch.tensor([tokenizer.encode(prompt_text)], dtype=torch.long)
generated = hope_model.generate(prompt_ids, max_new_tokens=100, temperature=0.8)
print('Prompt:', prompt_text)
print('Generated:', tokenizer.decode(generated[0].tolist()))

## 6. Surprise Evolution During Training

In [ ]:
# Plot how surprise changes over training steps
# We already collected this from the trainer
surprise_history = [l['surprise'] for l in trainer.train_epoch(loader, verbose=False)]

plt.figure(figsize=(10, 4))
plt.plot(surprise_history, alpha=0.5, linewidth=0.5)
# Smoothed
window = 50
smoothed = np.convolve(surprise_history, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(surprise_history)), smoothed, linewidth=2, label='Smoothed')
plt.xlabel('Step')
plt.ylabel('Mean Surprise')
plt.title('Surprise Evolution During Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('Surprise should generally decrease as the model learns to predict better.')